# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_04 — GMM-HMM Tick Regime Detection

### Research question

Can a Gaussian-mixture hidden Markov model detect microstructure regimes from L1 tick data using only best bid and best ask quotes?

This notebook follows naturally from:

```text
01_09_tick_to_bar_sampling_bias
03_03_markov_regime_switching
```

The previous regime-switching notebook used a Gaussian HMM on lower-frequency return and volatility features. This notebook moves to **tick-level L1 market data**, where emissions are noisier, heavier-tailed, and often multi-modal.

The realistic data constraint is:

```text
timestamp, bid1, ask1
```

Optionally:

```text
bid1_size, ask1_size
```

This notebook **does not assume full order-book depth**. It uses top-of-book features only:

1. mid-price returns;
2. quoted spread;
3. relative spread;
4. quote inter-arrival time;
5. quote intensity;
6. rolling micro-volatility;
7. optional top-of-book imbalance if sizes are available.

It covers:

1. synthetic L1 quote generation with hidden microstructure regimes;
2. L1 feature engineering without future leakage;
3. Gaussian HMM versus GMM-HMM motivation;
4. diagonal-covariance GMM-HMM implementation from scratch;
5. log-space forward-backward inference;
6. EM estimation;
7. Viterbi regime path;
8. regime posterior probabilities;
9. transition and duration diagnostics;
10. state/component interpretation;
11. out-of-sample online filtering;
12. a toy regime-based execution/risk filter;
13. limitations and research extensions.

The key idea:

> With only L1 bid/ask data, we cannot model the full limit order book, but we can still infer useful top-of-book microstructure regimes.

## 1. Why GMM-HMM for tick data?

A standard Gaussian HMM assumes:

$$
X_t \mid Z_t=k \sim \mathcal N(\mu_k,\Sigma_k)
$$

But tick-level features often have:

- heavy tails;
- many near-zero returns;
- occasional jumps;
- spread regime changes;
- irregular quote timing;
- mixtures of quiet and active behaviour inside the same state.

A GMM-HMM allows each hidden state to have a mixture of Gaussian components:

$$
p(X_t\mid Z_t=k) = \sum_{m=1}^{M} w_{k,m} \mathcal N(X_t;\mu_{k,m},\Sigma_{k,m})
$$

This gives each regime a richer emission distribution.

For this notebook, we use diagonal covariance matrices for stability and readability:

$$
\Sigma_{k,m} = \operatorname{diag}(\sigma^2_{k,m,1},\dots,\sigma^2_{k,m,d})
$$

## 2. L1 top-of-book features

With only:

$$
bid1_t,\quad ask1_t
$$

we can compute:

$$
mid_t=\frac{bid1_t+ask1_t}{2}
$$

$$
spread_t=ask1_t-bid1_t
$$

$$
relative\ spread_t=\frac{ask1_t-bid1_t}{mid_t}
$$

$$
r_t=\log(mid_t)-\log(mid_{t-1})
$$

For quote timing, if timestamps are irregular:

$$
\Delta t_t = timestamp_t - timestamp_{t-1}
$$

and a simple quote-intensity proxy is:

$$
intensity_t = \frac{1}{\Delta t_t+\epsilon}
$$

These are top-of-book features, not full LOB features.

Without deeper levels, we do **not** claim to know:

- depth imbalance across levels;
- queue position;
- hidden liquidity;
- cancellation flow;
- market-order flow;
- full order-book pressure.

## 3. Hidden microstructure regimes

We simulate four regimes:

| Regime | Interpretation |
|---:|---|
| 0 | quiet tight-spread |
| 1 | normal liquid |
| 2 | active volatile |
| 3 | stressed wide-spread |

In real futures data, such regimes could correspond to:

- open/close bursts;
- pre-news stress;
- lunch/night-session liquidity changes;
- contract roll liquidity migration;
- price-limit proximity;
- product-specific microstructure.

The exact labels are learned after fitting and sorted by inferred microstructure risk.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class TickGMMHMMConfig:
    n_ticks: int = 30_000
    n_states: int = 4
    n_components: int = 2
    train_fraction: float = 0.70
    tick_size: float = 1.0
    base_price: float = 3000.0
    rolling_window: int = 100
    max_em_iter: int = 35
    min_variance: float = 1e-4
    seed: int = 42


config = TickGMMHMMConfig()
config

## 5. Simulating L1 tick data

The synthetic process has:

1. a hidden Markov chain for microstructure regimes;
2. regime-dependent mid-price volatility;
3. regime-dependent spread;
4. regime-dependent quote intensity;
5. optional best-bid/best-ask sizes.

This gives us a controlled dataset where the true hidden state is known.

In [ ]:
def simulate_l1_tick_data(config: TickGMMHMMConfig) -> tuple[pd.DataFrame, np.ndarray]:
    rng = np.random.default_rng(config.seed)
    n = config.n_ticks
    K = config.n_states

    # Persistent but switchable transition matrix.
    transition = np.array([
        [0.985, 0.013, 0.002, 0.000],
        [0.010, 0.970, 0.018, 0.002],
        [0.003, 0.025, 0.950, 0.022],
        [0.002, 0.010, 0.060, 0.928],
    ])

    # Per-tick log-return volatility by regime.
    ret_vol = np.array([0.00005, 0.00012, 0.00028, 0.00060])
    ret_mean = np.array([0.0, 0.0, 0.000005, -0.000020])

    # Quote interval scale in seconds. Stress and active regimes update faster.
    interval_scale = np.array([5.0, 2.0, 0.7, 0.35])

    # Spread ticks distribution by regime.
    spread_tick_probs = {
        0: ([1, 2], [0.90, 0.10]),
        1: ([1, 2, 3], [0.65, 0.30, 0.05]),
        2: ([1, 2, 3, 4], [0.35, 0.35, 0.20, 0.10]),
        3: ([2, 3, 4, 5, 6], [0.20, 0.25, 0.25, 0.20, 0.10]),
    }

    states = np.zeros(n, dtype=int)
    log_mid = np.zeros(n)
    intervals = np.zeros(n)
    spread_ticks = np.zeros(n, dtype=int)

    states[0] = 0
    log_mid[0] = np.log(config.base_price)

    for t in range(1, n):
        states[t] = rng.choice(K, p=transition[states[t - 1]])

        # Occasional micro-jump in stress/active regimes.
        jump = 0.0
        if states[t] >= 2 and rng.uniform() < 0.002:
            jump = rng.normal(0.0, 4.0 * ret_vol[states[t]])

        log_mid[t] = (
            log_mid[t - 1]
            + ret_mean[states[t]]
            + ret_vol[states[t]] * rng.standard_normal()
            + jump
        )

        intervals[t] = rng.exponential(interval_scale[states[t]])

        ticks, probs = spread_tick_probs[states[t]]
        spread_ticks[t] = rng.choice(ticks, p=probs)

    intervals[0] = intervals[1]
    spread_ticks[0] = spread_ticks[1]

    mid = np.exp(log_mid)
    mid_rounded = np.round(mid / config.tick_size) * config.tick_size

    spread = spread_ticks * config.tick_size
    bid1 = np.round((mid_rounded - spread / 2) / config.tick_size) * config.tick_size
    ask1 = bid1 + spread

    # Optional top-of-book sizes.
    bid1_size = rng.lognormal(mean=3.2 - 0.20 * states, sigma=0.55, size=n).round()
    ask1_size = rng.lognormal(mean=3.2 + 0.10 * states, sigma=0.55, size=n).round()

    timestamps = pd.Timestamp("2024-01-02 09:00:00") + pd.to_timedelta(np.cumsum(intervals), unit="s")

    df = pd.DataFrame({
        "timestamp": timestamps,
        "true_state": states,
        "bid1": bid1.astype(float),
        "ask1": ask1.astype(float),
        "bid1_size": bid1_size.astype(float),
        "ask1_size": ask1_size.astype(float),
        "quote_interval_seconds": intervals,
        "true_mid": mid,
        "true_transition_state": states
    })

    return df, transition


ticks_raw, true_transition = simulate_l1_tick_data(config)

ticks_raw.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(ticks_raw["timestamp"], ticks_raw["true_mid"])
plt.title("Synthetic L1 Mid Price")
plt.xlabel("Timestamp")
plt.ylabel("Mid price")
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(ticks_raw["timestamp"], ticks_raw["true_state"], drawstyle="steps-post")
plt.title("True Hidden Microstructure State")
plt.xlabel("Timestamp")
plt.ylabel("State")
plt.yticks([0, 1, 2, 3])
plt.show()

## 6. L1 feature engineering

We compute features that are available at or before each tick.

Important no-leakage principle:

> Rolling features must use current/past ticks only. No centred windows.

Features:

1. mid return;
2. absolute mid return;
3. spread ticks;
4. relative spread;
5. log quote interval;
6. quote intensity;
7. rolling micro-volatility;
8. rolling average spread;
9. optional top-of-book imbalance.

In [ ]:
def add_l1_features(df: pd.DataFrame, config: TickGMMHMMConfig) -> pd.DataFrame:
    out = df.sort_values("timestamp").copy().reset_index(drop=True)

    out["mid"] = (out["bid1"] + out["ask1"]) / 2.0
    out["spread"] = out["ask1"] - out["bid1"]
    out["spread_ticks"] = out["spread"] / config.tick_size
    out["relative_spread"] = out["spread"] / out["mid"].clip(lower=1e-12)

    out["log_mid"] = np.log(out["mid"].clip(lower=1e-12))
    out["mid_return"] = out["log_mid"].diff().fillna(0.0)
    out["abs_mid_return"] = out["mid_return"].abs()

    out["quote_interval_seconds"] = out["timestamp"].diff().dt.total_seconds().fillna(out["quote_interval_seconds"].median())
    out["quote_interval_seconds"] = out["quote_interval_seconds"].clip(lower=1e-3)
    out["log_quote_interval"] = np.log1p(out["quote_interval_seconds"])
    out["quote_intensity"] = 1.0 / out["quote_interval_seconds"]

    if {"bid1_size", "ask1_size"}.issubset(out.columns):
        denom = out["bid1_size"] + out["ask1_size"]
        out["top_of_book_imbalance"] = np.where(
            denom > 0,
            (out["bid1_size"] - out["ask1_size"]) / denom,
            0.0
        )
    else:
        out["top_of_book_imbalance"] = 0.0

    w = config.rolling_window

    out["rolling_micro_vol"] = out["mid_return"].rolling(w).std().fillna(method="bfill")
    out["rolling_abs_return"] = out["abs_mid_return"].rolling(w).mean().fillna(method="bfill")
    out["rolling_spread_ticks"] = out["spread_ticks"].rolling(w).mean().fillna(method="bfill")
    out["rolling_quote_intensity"] = out["quote_intensity"].rolling(w).mean().fillna(method="bfill")

    return out.dropna().reset_index(drop=True)


ticks = add_l1_features(ticks_raw, config)

ticks.head()

In [ ]:
feature_cols_raw = [
    "mid_return",
    "abs_mid_return",
    "spread_ticks",
    "log_quote_interval",
    "quote_intensity",
    "rolling_micro_vol",
    "rolling_spread_ticks",
    "rolling_quote_intensity",
    "top_of_book_imbalance"
]

feature_summary = ticks[feature_cols_raw].describe().T

feature_summary

## 7. Standardisation

GMM-HMM estimation is numerically easier when features are standardised.

We use training-set mean and standard deviation only.

This avoids leaking test-period information into the feature scaling.

In [ ]:
split_idx = int(len(ticks) * config.train_fraction)

train_ticks = ticks.iloc[:split_idx].copy()
test_ticks = ticks.iloc[split_idx:].copy()

feature_means = train_ticks[feature_cols_raw].mean()
feature_stds = train_ticks[feature_cols_raw].std(ddof=1).replace(0, 1.0)

feature_cols = []

for col in feature_cols_raw:
    z_col = f"{col}_z"
    ticks[z_col] = (ticks[col] - feature_means[col]) / feature_stds[col]
    train_ticks[z_col] = (train_ticks[col] - feature_means[col]) / feature_stds[col]
    test_ticks[z_col] = (test_ticks[col] - feature_means[col]) / feature_stds[col]
    feature_cols.append(z_col)

X_train = train_ticks[feature_cols].to_numpy()
X_test = test_ticks[feature_cols].to_numpy()

pd.Series({
    "n_total_ticks": len(ticks),
    "n_train_ticks": len(train_ticks),
    "n_test_ticks": len(test_ticks),
    "n_features": len(feature_cols)
})

## 8. Log-space helpers

We implement:

1. log-sum-exp;
2. diagonal Gaussian log densities;
3. forward-backward inference;
4. Viterbi decoding.

Everything is written from scratch to make the notebook repo-independent.

In [ ]:
def logsumexp(a, axis=None, keepdims=False):
    a = np.asarray(a, dtype=float)
    m = np.max(a, axis=axis, keepdims=True)
    out = m + np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)
    return out


def normalize_rows(A, floor=1e-12):
    A = np.maximum(np.asarray(A, dtype=float), floor)
    return A / A.sum(axis=1, keepdims=True)


def diagonal_gaussian_logpdf(X, means, variances):
    """
    X shape: (T, d)
    means shape: (K, M, d)
    variances shape: (K, M, d)

    Returns: (T, K, M)
    """
    X = np.asarray(X, dtype=float)
    T, d = X.shape
    K, M, _ = means.shape

    out = np.empty((T, K, M), dtype=float)

    for k in range(K):
        for m in range(M):
            var = np.maximum(variances[k, m], 1e-10)
            diff = X - means[k, m]
            out[:, k, m] = -0.5 * (
                d * np.log(2 * np.pi)
                + np.sum(np.log(var))
                + np.sum(diff**2 / var, axis=1)
            )

    return out


def mixture_log_emissions(X, weights, means, variances):
    """
    State log emissions for GMM-HMM.

    weights shape: (K, M)
    returns:
        log_emissions shape (T, K)
        log_component_density shape (T, K, M)
    """
    log_comp_density = diagonal_gaussian_logpdf(X, means, variances)
    log_weights = np.log(np.maximum(weights, 1e-300))[None, :, :]
    log_emissions = logsumexp(log_weights + log_comp_density, axis=2)
    return log_emissions, log_comp_density

In [ ]:
def forward_log(log_emissions, pi, A):
    T, K = log_emissions.shape
    log_pi = np.log(np.maximum(pi, 1e-300))
    log_A = np.log(np.maximum(A, 1e-300))

    log_alpha = np.empty((T, K))
    log_alpha[0] = log_pi + log_emissions[0]

    for t in range(1, T):
        log_alpha[t] = log_emissions[t] + logsumexp(log_alpha[t - 1][:, None] + log_A, axis=0)

    ll = float(logsumexp(log_alpha[-1], axis=0))
    return log_alpha, ll


def backward_log(log_emissions, A):
    T, K = log_emissions.shape
    log_A = np.log(np.maximum(A, 1e-300))

    log_beta = np.zeros((T, K))

    for t in range(T - 2, -1, -1):
        log_beta[t] = logsumexp(log_A + log_emissions[t + 1][None, :] + log_beta[t + 1][None, :], axis=1)

    return log_beta


def forward_backward_gmmhmm(log_emissions, pi, A):
    log_alpha, ll = forward_log(log_emissions, pi, A)
    log_beta = backward_log(log_emissions, A)

    log_gamma = log_alpha + log_beta - ll
    gamma = np.exp(log_gamma)

    T, K = log_emissions.shape
    log_A = np.log(np.maximum(A, 1e-300))
    xi = np.empty((T - 1, K, K))

    for t in range(T - 1):
        log_xi_t = (
            log_alpha[t][:, None]
            + log_A
            + log_emissions[t + 1][None, :]
            + log_beta[t + 1][None, :]
            - ll
        )
        xi[t] = np.exp(log_xi_t)

    return {
        "log_alpha": log_alpha,
        "log_beta": log_beta,
        "gamma": gamma,
        "xi": xi,
        "log_likelihood": ll
    }


def viterbi_log(log_emissions, pi, A):
    T, K = log_emissions.shape
    log_pi = np.log(np.maximum(pi, 1e-300))
    log_A = np.log(np.maximum(A, 1e-300))

    delta = np.empty((T, K))
    psi = np.zeros((T, K), dtype=int)

    delta[0] = log_pi + log_emissions[0]

    for t in range(1, T):
        scores = delta[t - 1][:, None] + log_A
        psi[t] = np.argmax(scores, axis=0)
        delta[t] = log_emissions[t] + np.max(scores, axis=0)

    states = np.zeros(T, dtype=int)
    states[-1] = np.argmax(delta[-1])

    for t in range(T - 2, -1, -1):
        states[t] = psi[t + 1, states[t + 1]]

    return states

## 9. Initialisation

We initialise states using a microstructure risk score:

$$
\begin{aligned}
score_t &= z(\text{rolling vol}) \\
&\quad + z(\text{spread}) \\
&\quad + z(\text{quote intensity}) \\
&\quad + z(|r_t|)
\end{aligned}
$$

Then split the score into quantile buckets.

Within each state, mixture components are initialised by splitting observations using absolute return.

In [ ]:
def initialise_gmmhmm(X, train_df, feature_cols, K, M, min_variance=1e-4):
    T, d = X.shape

    risk_score = (
        train_df["rolling_micro_vol_z"].to_numpy()
        + train_df["rolling_spread_ticks_z"].to_numpy()
        + train_df["rolling_quote_intensity_z"].to_numpy()
        + train_df["abs_mid_return_z"].to_numpy()
    )

    state_init = pd.qcut(risk_score, q=K, labels=False, duplicates="drop").astype(int)
    state_init = np.asarray(state_init)

    means = np.zeros((K, M, d))
    variances = np.zeros((K, M, d))
    weights = np.ones((K, M)) / M

    for k in range(K):
        state_mask = state_init == k

        if state_mask.sum() < M * 5:
            state_mask = np.ones(T, dtype=bool)

        X_state = X[state_mask]
        absret_state = train_df.loc[state_mask, "abs_mid_return"].to_numpy()

        comp_init = pd.qcut(absret_state, q=M, labels=False, duplicates="drop").astype(int)

        for m in range(M):
            comp_mask = comp_init == m

            if np.sum(comp_mask) < 5:
                comp_mask = np.ones(len(X_state), dtype=bool)

            X_comp = X_state[comp_mask]
            means[k, m] = X_comp.mean(axis=0)
            variances[k, m] = np.maximum(X_comp.var(axis=0), min_variance)
            weights[k, m] = len(X_comp)

        weights[k] = weights[k] / weights[k].sum()

    pi = np.ones(K) / K

    A = np.full((K, K), 0.03 / (K - 1))
    np.fill_diagonal(A, 0.97)
    A = normalize_rows(A)

    return pi, A, weights, means, variances


pi0, A0, weights0, means0, vars0 = initialise_gmmhmm(
    X_train,
    train_ticks,
    feature_cols,
    config.n_states,
    config.n_components,
    min_variance=config.min_variance
)

pd.DataFrame(A0)

## 10. EM estimation for diagonal-covariance GMM-HMM

The E-step computes:

$$
\gamma_t(k)=p(Z_t=k\mid X_{1:T})
$$

and component responsibilities:

$$
\eta_t(k,m) = p(Z_t=k, C_t=m\mid X_{1:T})
$$

where $C_t$ is the mixture component inside the state.

The M-step updates:

- initial probabilities $\pi$;
- transition matrix $A$;
- mixture weights $w_{k,m}$;
- component means;
- diagonal variances.

We use diagonal covariance matrices to keep estimation stable for tick features.

In [ ]:
def fit_gmmhmm_diag(
    X,
    pi_init,
    A_init,
    weights_init,
    means_init,
    variances_init,
    max_iter=35,
    tol=1e-4,
    min_variance=1e-4
):
    X = np.asarray(X, dtype=float)
    T, d = X.shape
    K, M, _ = means_init.shape

    pi = pi_init.copy()
    A = A_init.copy()
    weights = weights_init.copy()
    means = means_init.copy()
    variances = variances_init.copy()

    loglik_history = []

    for iteration in range(max_iter):
        log_emissions, log_comp_density = mixture_log_emissions(X, weights, means, variances)
        fb = forward_backward_gmmhmm(log_emissions, pi, A)

        gamma = fb["gamma"]
        xi = fb["xi"]
        ll = fb["log_likelihood"]
        loglik_history.append(ll)

        # Component posterior given state and observation.
        log_weights = np.log(np.maximum(weights, 1e-300))[None, :, :]
        log_comp_posterior = log_weights + log_comp_density - log_emissions[:, :, None]
        comp_posterior = np.exp(log_comp_posterior)

        eta = gamma[:, :, None] * comp_posterior

        # Update pi and A.
        pi = gamma[0] / gamma[0].sum()
        A = xi.sum(axis=0) / np.maximum(gamma[:-1].sum(axis=0)[:, None], 1e-12)
        A = normalize_rows(A)

        # Update mixture parameters.
        for k in range(K):
            gamma_k_sum = np.sum(gamma[:, k])

            for m in range(M):
                w_km = eta[:, k, m]
                denom = np.sum(w_km)

                if denom < 1e-8:
                    continue

                means[k, m] = (w_km[:, None] * X).sum(axis=0) / denom
                diff = X - means[k, m]
                variances[k, m] = np.maximum((w_km[:, None] * diff**2).sum(axis=0) / denom, min_variance)
                weights[k, m] = denom / max(gamma_k_sum, 1e-12)

            weights[k] = weights[k] / weights[k].sum()

        if iteration > 0 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

    log_emissions, log_comp_density = mixture_log_emissions(X, weights, means, variances)
    fb = forward_backward_gmmhmm(log_emissions, pi, A)

    return {
        "pi": pi,
        "A": A,
        "weights": weights,
        "means": means,
        "variances": variances,
        "loglik_history": loglik_history,
        "posterior": fb["gamma"],
        "log_likelihood": fb["log_likelihood"],
        "log_emissions": log_emissions,
        "n_iter": len(loglik_history)
    }


gmmhmm_fit = fit_gmmhmm_diag(
    X_train,
    pi0,
    A0,
    weights0,
    means0,
    vars0,
    max_iter=config.max_em_iter,
    tol=1e-4,
    min_variance=config.min_variance
)

pd.Series({
    "log_likelihood": gmmhmm_fit["log_likelihood"],
    "n_iter": gmmhmm_fit["n_iter"]
})

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(gmmhmm_fit["loglik_history"], marker="o")
plt.title("GMM-HMM EM Log-Likelihood")
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.show()

## 11. Sort states by microstructure risk

HMM labels are arbitrary.

We sort states by posterior-weighted microstructure risk:

$$
risk_k = E[ |r_t|+ spread_t+ rolling\ vol_t+ rolling\ intensity_t \mid Z_t=k ]
$$

This makes state labels interpretable from calm to stressed.

In [ ]:
def state_risk_scores(train_df, posterior):
    risk_raw = (
        train_df["abs_mid_return_z"].to_numpy()
        + train_df["rolling_micro_vol_z"].to_numpy()
        + train_df["rolling_spread_ticks_z"].to_numpy()
        + train_df["rolling_quote_intensity_z"].to_numpy()
    )

    scores = []
    for k in range(posterior.shape[1]):
        w = posterior[:, k]
        scores.append(float(np.sum(w * risk_raw) / max(np.sum(w), 1e-12)))

    return np.array(scores)


risk_scores = state_risk_scores(train_ticks, gmmhmm_fit["posterior"])
state_order = np.argsort(risk_scores)

state_order, risk_scores

In [ ]:
def reorder_gmmhmm_fit(fit, order):
    order = np.asarray(order, dtype=int)

    return {
        "pi": fit["pi"][order],
        "A": fit["A"][np.ix_(order, order)],
        "weights": fit["weights"][order],
        "means": fit["means"][order],
        "variances": fit["variances"][order],
        "posterior": fit["posterior"][:, order],
        "log_likelihood": fit["log_likelihood"],
        "loglik_history": fit["loglik_history"],
        "n_iter": fit["n_iter"]
    }


gmmhmm_sorted = reorder_gmmhmm_fit(gmmhmm_fit, state_order)

regime_labels = ["quiet", "normal", "active", "stressed"]

train_posterior = gmmhmm_sorted["posterior"]
train_state = train_posterior.argmax(axis=1)

pd.Series(train_state).value_counts().sort_index()

## 12. Viterbi regime path

The posterior most likely state chooses the best state at each tick individually.

The Viterbi path finds the most likely full state sequence.

This is useful for regime labelling and segment analysis.

In [ ]:
train_log_emissions, _ = mixture_log_emissions(
    X_train,
    gmmhmm_sorted["weights"],
    gmmhmm_sorted["means"],
    gmmhmm_sorted["variances"]
)

train_viterbi = viterbi_log(
    train_log_emissions,
    gmmhmm_sorted["pi"],
    gmmhmm_sorted["A"]
)

pd.Series(train_viterbi).value_counts().sort_index()

## 13. Training regime diagnostics

We inspect:

- true synthetic state;
- inferred Viterbi state;
- posterior probabilities.

In real data, the true state is unavailable, so we rely on regime statistics and economic interpretation.

In [ ]:
train_regimes = train_ticks[[
    "timestamp",
    "true_state",
    "mid",
    "mid_return",
    "spread_ticks",
    "quote_interval_seconds",
    "quote_intensity",
    "rolling_micro_vol",
    "top_of_book_imbalance"
]].copy()

train_regimes["posterior_state"] = train_state
train_regimes["viterbi_state"] = train_viterbi

for k, label in enumerate(regime_labels):
    train_regimes[f"prob_{label}"] = train_posterior[:, k]

train_regimes.head()

In [ ]:
sample_plot = train_regimes.iloc[:5000]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(sample_plot["timestamp"], sample_plot["mid"])
axes[0].set_title("Training Mid Price Sample")

axes[1].plot(sample_plot["timestamp"], sample_plot["true_state"], drawstyle="steps-post")
axes[1].set_title("True Synthetic State")

axes[2].plot(sample_plot["timestamp"], sample_plot["viterbi_state"], drawstyle="steps-post")
axes[2].set_title("GMM-HMM Viterbi State")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sample_plot = train_regimes.iloc[:5000]
for label in regime_labels:
    plt.plot(sample_plot["timestamp"], sample_plot[f"prob_{label}"], label=label)
plt.title("Training Posterior Regime Probabilities")
plt.xlabel("Timestamp")
plt.ylabel("Probability")
plt.legend()
plt.show()

## 14. Transition matrix and expected duration

The transition matrix:

$$
A_{ij}=P(Z_t=j\mid Z_{t-1}=i)
$$

summarises regime persistence.

Expected duration:

$$
E[D_i]=\frac{1}{1-A_{ii}}
$$

when $A_{ii}<1$.

In [ ]:
transition_df = pd.DataFrame(
    gmmhmm_sorted["A"],
    columns=[f"to_{x}" for x in regime_labels],
    index=[f"from_{x}" for x in regime_labels]
)

transition_df

In [ ]:
duration_report = []

for k, label in enumerate(regime_labels):
    p_stay = gmmhmm_sorted["A"][k, k]
    duration_report.append({
        "regime": label,
        "self_transition_probability": p_stay,
        "expected_duration_ticks": 1.0 / max(1e-12, 1.0 - p_stay)
    })

duration_report = pd.DataFrame(duration_report)

duration_report

## 15. Regime interpretation

For each inferred state, compute posterior-weighted top-of-book statistics:

- return volatility;
- average spread;
- quote interval;
- quote intensity;
- top-of-book imbalance.

In [ ]:
def posterior_weighted_regime_stats(df, posterior, labels):
    rows = []

    for k, label in enumerate(labels):
        w = posterior[:, k]
        w_sum = w.sum()

        def wavg(x):
            return float(np.sum(w * x) / max(w_sum, 1e-12))

        mean_ret = wavg(df["mid_return"].to_numpy())
        var_ret = wavg((df["mid_return"].to_numpy() - mean_ret) ** 2)

        rows.append({
            "regime": label,
            "posterior_weight": float(w_sum),
            "mean_mid_return": mean_ret,
            "mid_return_vol": sqrt(var_ret),
            "mean_abs_mid_return": wavg(df["abs_mid_return"].to_numpy()),
            "mean_spread_ticks": wavg(df["spread_ticks"].to_numpy()),
            "mean_quote_interval_seconds": wavg(df["quote_interval_seconds"].to_numpy()),
            "mean_quote_intensity": wavg(df["quote_intensity"].to_numpy()),
            "mean_rolling_micro_vol": wavg(df["rolling_micro_vol"].to_numpy()),
            "mean_top_of_book_imbalance": wavg(df["top_of_book_imbalance"].to_numpy())
        })

    return pd.DataFrame(rows)


regime_stats = posterior_weighted_regime_stats(train_ticks, train_posterior, regime_labels)

regime_stats

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(regime_stats["regime"], regime_stats["mean_spread_ticks"])
plt.title("Mean Spread by Inferred Regime")
plt.xlabel("Regime")
plt.ylabel("Spread in ticks")
plt.show()

plt.figure(figsize=(10, 6))
plt.bar(regime_stats["regime"], regime_stats["mean_rolling_micro_vol"])
plt.title("Mean Rolling Micro-Volatility by Regime")
plt.xlabel("Regime")
plt.ylabel("Rolling micro-volatility")
plt.show()

## 16. Mixture component diagnostics

Inside each hidden state, the GMM components can capture substructure.

For example:

- low-shock vs high-shock observations inside the same regime;
- tighter vs wider spread inside a liquidity regime;
- normal vs jump-like observations.

We inspect mixture weights and component means.

In [ ]:
component_rows = []

for k, label in enumerate(regime_labels):
    for m in range(config.n_components):
        row = {
            "regime": label,
            "component": m,
            "mixture_weight": gmmhmm_sorted["weights"][k, m]
        }

        for j, col in enumerate(feature_cols):
            row[f"mean_{col}"] = gmmhmm_sorted["means"][k, m, j]
            row[f"std_{col}"] = np.sqrt(gmmhmm_sorted["variances"][k, m, j])

        component_rows.append(row)

component_report = pd.DataFrame(component_rows)

component_report

## 17. Confusion diagnostic on synthetic states

Because this is synthetic data, we can compare inferred regimes with true hidden states.

In real data, there is no true state, so this diagnostic is unavailable.

In [ ]:
def confusion_table(true_state, inferred_state):
    return pd.crosstab(
        pd.Series(true_state, name="true_state"),
        pd.Series(inferred_state, name="inferred_state"),
        normalize="index"
    )


train_confusion_posterior = confusion_table(train_regimes["true_state"], train_regimes["posterior_state"])
train_confusion_viterbi = confusion_table(train_regimes["true_state"], train_regimes["viterbi_state"])

train_confusion_posterior

In [ ]:
train_confusion_viterbi

## 18. Out-of-sample online filtering

For live use, we cannot use future ticks.

We filter online:

$$
P(Z_t\mid X_{1:t})
$$

using only observations up to tick $t$.

The test set uses the fitted training parameters.

In [ ]:
def filter_gmmhmm_online(X, pi, A, weights, means, variances):
    log_emissions, _ = mixture_log_emissions(X, weights, means, variances)
    T, K = log_emissions.shape

    log_A = np.log(np.maximum(A, 1e-300))
    log_alpha = np.empty((T, K))
    filtered = np.empty((T, K))

    log_alpha[0] = np.log(np.maximum(pi, 1e-300)) + log_emissions[0]
    log_alpha[0] -= logsumexp(log_alpha[0])
    filtered[0] = np.exp(log_alpha[0])

    for t in range(1, T):
        prediction = logsumexp(log_alpha[t - 1][:, None] + log_A, axis=0)
        log_alpha[t] = prediction + log_emissions[t]
        log_alpha[t] -= logsumexp(log_alpha[t])
        filtered[t] = np.exp(log_alpha[t])

    return filtered, log_emissions


last_train_posterior = train_posterior[-1]

test_filtered, test_log_emissions = filter_gmmhmm_online(
    X_test,
    pi=last_train_posterior,
    A=gmmhmm_sorted["A"],
    weights=gmmhmm_sorted["weights"],
    means=gmmhmm_sorted["means"],
    variances=gmmhmm_sorted["variances"]
)

test_state = test_filtered.argmax(axis=1)

test_regimes = test_ticks[[
    "timestamp",
    "true_state",
    "mid",
    "mid_return",
    "spread_ticks",
    "quote_interval_seconds",
    "quote_intensity",
    "rolling_micro_vol",
    "top_of_book_imbalance"
]].copy()

test_regimes["filtered_state"] = test_state

for k, label in enumerate(regime_labels):
    test_regimes[f"prob_{label}"] = test_filtered[:, k]

test_regimes.head()

In [ ]:
sample_test = test_regimes.iloc[:5000]

plt.figure(figsize=(12, 6))
for label in regime_labels:
    plt.plot(sample_test["timestamp"], sample_test[f"prob_{label}"], label=label)
plt.title("Out-of-Sample Filtered GMM-HMM Regime Probabilities")
plt.xlabel("Timestamp")
plt.ylabel("Filtered probability")
plt.legend()
plt.show()

## 19. Test-period regime statistics

We compute realised microstructure statistics conditional on filtered state.

This checks whether the regimes are economically meaningful out of sample.

In [ ]:
test_state_stats = (
    test_regimes
    .groupby("filtered_state")
    .agg(
        n_ticks=("mid_return", "count"),
        mean_return=("mid_return", "mean"),
        return_vol=("mid_return", "std"),
        mean_abs_return=("mid_return", lambda x: np.mean(np.abs(x))),
        mean_spread_ticks=("spread_ticks", "mean"),
        mean_quote_interval_seconds=("quote_interval_seconds", "mean"),
        mean_quote_intensity=("quote_intensity", "mean"),
        mean_rolling_micro_vol=("rolling_micro_vol", "mean"),
        mean_top_of_book_imbalance=("top_of_book_imbalance", "mean")
    )
    .reset_index()
)

test_state_stats["regime"] = test_state_stats["filtered_state"].map(dict(enumerate(regime_labels)))

test_state_stats

## 20. Toy regime-based execution/risk filter

A simple use case:

- trade normally in quiet/normal regimes;
- reduce exposure in active regimes;
- stop trading in stressed regimes.

This is **not** an alpha claim.

It demonstrates how a tick-regime model can control execution participation or risk exposure.

We define position:

$$
w_t =
\begin{cases}
1.0, & P(stressed)_t < 0.30 \text{ and } P(active)_t < 0.55 \\
0.5, & P(stressed)_t < 0.50 \\
0.0, & \text{otherwise}
\end{cases}
$$

We apply the position to next-tick mid returns and subtract a simple spread cost when position changes.

In [ ]:
def toy_tick_regime_filter(test_regimes, spread_cost_multiplier=0.5):
    out = test_regimes.copy()

    p_active = out["prob_active"].to_numpy()
    p_stress = out["prob_stressed"].to_numpy()

    position = np.where(
        (p_stress < 0.30) & (p_active < 0.55),
        1.0,
        np.where(p_stress < 0.50, 0.5, 0.0)
    )

    out["position"] = position
    out["next_mid_return"] = out["mid_return"].shift(-1).fillna(0.0)

    # Spread cost proxy when changing position.
    position_change = np.abs(np.diff(np.r_[position[0], position]))
    relative_spread = test_ticks.loc[out.index + test_ticks.index.min(), "relative_spread"].to_numpy() if "relative_spread" in test_ticks else np.zeros(len(out))

    out["transaction_cost_proxy"] = spread_cost_multiplier * position_change * relative_spread
    out["strategy_return"] = out["position"] * out["next_mid_return"] - out["transaction_cost_proxy"]
    out["always_on_return"] = out["next_mid_return"]

    out["cum_strategy"] = np.exp(out["strategy_return"].cumsum())
    out["cum_always_on"] = np.exp(out["always_on_return"].cumsum())

    return out


strategy_df = toy_tick_regime_filter(test_regimes)

strategy_summary = pd.Series({
    "mean_position": strategy_df["position"].mean(),
    "turnover_proxy": strategy_df["position"].diff().abs().fillna(0).mean(),
    "strategy_total_log_return": strategy_df["strategy_return"].sum(),
    "always_on_total_log_return": strategy_df["always_on_return"].sum(),
    "strategy_return_vol": strategy_df["strategy_return"].std(ddof=1),
    "always_on_return_vol": strategy_df["always_on_return"].std(ddof=1),
    "mean_transaction_cost_proxy": strategy_df["transaction_cost_proxy"].mean()
})

strategy_summary

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(strategy_df["timestamp"], strategy_df["cum_strategy"], label="Regime-filtered exposure")
plt.plot(strategy_df["timestamp"], strategy_df["cum_always_on"], label="Always-on exposure")
plt.title("Toy Tick-Regime Filter")
plt.xlabel("Timestamp")
plt.ylabel("Cumulative growth proxy")
plt.legend()
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(strategy_df["timestamp"].iloc[:5000], strategy_df["position"].iloc[:5000], drawstyle="steps-post")
plt.title("Sample Regime-Based Exposure")
plt.xlabel("Timestamp")
plt.ylabel("Position")
plt.ylim(-0.05, 1.05)
plt.show()

## 21. Practical checklist for L1 GMM-HMM regime detection

Before trusting tick-regime output, check:

1. **Data scope**  
   Are features truly L1-only?

2. **No leakage**  
   Are rolling features backward-looking only?

3. **Session structure**  
   Are day/night/lunch breaks handled?

4. **Feature scaling**  
   Are train-set means and standard deviations used?

5. **State count**  
   Is the number of regimes interpretable?

6. **Mixture count**  
   Does each state need multiple components?

7. **Log-space inference**  
   Is the forward-backward algorithm numerically stable?

8. **State sorting**  
   Are regimes sorted after fitting?

9. **Economic interpretation**  
   Do states differ in spread, intensity, and micro-volatility?

10. **Out-of-sample filtering**  
   Are live probabilities filtered, not smoothed?

11. **Execution costs**  
   Are spread and turnover considered?

12. **Robustness**  
   Do regimes persist across products, sessions, and time periods?

## 22. Saving outputs

The notebook saves:

1. synthetic L1 tick data;
2. engineered feature table;
3. GMM-HMM parameters;
4. transition matrix;
5. regime statistics;
6. mixture component diagnostics;
7. train posterior probabilities;
8. test filtered probabilities;
9. confusion tables;
10. toy strategy output;
11. manifest.

In [ ]:
output_dir = Path("data/processed/gmm_hmm_tick_regime_detection_v1")
output_dir.mkdir(parents=True, exist_ok=True)

raw_ticks_path = output_dir / "synthetic_l1_ticks.csv"
features_path = output_dir / "l1_tick_features.csv"
transition_path = output_dir / "gmmhmm_transition_matrix.csv"
duration_path = output_dir / "duration_report.csv"
regime_stats_path = output_dir / "regime_statistics.csv"
component_report_path = output_dir / "component_report.csv"
train_regimes_path = output_dir / "train_regime_probabilities.csv"
test_regimes_path = output_dir / "test_filtered_regime_probabilities.csv"
strategy_path = output_dir / "toy_tick_regime_filter.csv"
strategy_summary_path = output_dir / "strategy_summary.csv"
train_confusion_path = output_dir / "train_confusion_viterbi.csv"
params_path = output_dir / "gmmhmm_parameters.json"
manifest_path = output_dir / "manifest.json"

ticks_raw.to_csv(raw_ticks_path, index=False)
ticks.to_csv(features_path, index=False)
transition_df.to_csv(transition_path)
duration_report.to_csv(duration_path, index=False)
regime_stats.to_csv(regime_stats_path, index=False)
component_report.to_csv(component_report_path, index=False)
train_regimes.to_csv(train_regimes_path, index=False)
test_regimes.to_csv(test_regimes_path, index=False)
strategy_df.to_csv(strategy_path, index=False)
strategy_summary.to_frame("value").to_csv(strategy_summary_path)
train_confusion_viterbi.to_csv(train_confusion_path)

params_payload = {
    "pi": gmmhmm_sorted["pi"].tolist(),
    "A": gmmhmm_sorted["A"].tolist(),
    "weights": gmmhmm_sorted["weights"].tolist(),
    "means": gmmhmm_sorted["means"].tolist(),
    "variances": gmmhmm_sorted["variances"].tolist(),
    "feature_cols": feature_cols,
    "feature_means": feature_means.to_dict(),
    "feature_stds": feature_stds.to_dict(),
    "regime_labels": regime_labels,
    "log_likelihood": float(gmmhmm_sorted["log_likelihood"]),
    "n_iter": int(gmmhmm_sorted["n_iter"])
}

params_path.write_text(json.dumps(params_payload, indent=2, default=str))

manifest = {
    "dataset_name": "gmm_hmm_tick_regime_detection_outputs",
    "schema_version": "gmm_hmm_tick_regime_detection_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "feature_cols": feature_cols,
    "regime_labels": regime_labels,
    "log_likelihood": float(gmmhmm_sorted["log_likelihood"]),
    "n_iter": int(gmmhmm_sorted["n_iter"]),
    "strategy_summary": strategy_summary.to_dict(),
    "notes": [
        "Synthetic data contains L1 top-of-book bid1 and ask1 quotes.",
        "No full limit order book is assumed.",
        "Features are based on mid returns, spread, quote timing, rolling micro-volatility, and optional top-of-book imbalance.",
        "GMM-HMM uses diagonal covariance Gaussian mixture emissions.",
        "States are sorted by posterior-weighted microstructure risk.",
        "Test probabilities are filtered online using fitted training parameters.",
        "Toy strategy is a risk/execution filter demonstration, not an alpha claim."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

raw_ticks_path, features_path, transition_path, test_regimes_path, manifest_path

## 23. Limitations

### 23.1 Synthetic data

This notebook uses synthetic L1 data.

Real tick data has exchange-specific quirks, missing quotes, session breaks, stale quotes, crossed markets, and product-specific trading calendars.

### 23.2 L1 only

The model only uses best bid and best ask.

It cannot infer full depth, queue position, hidden liquidity, or cancellation pressure.

### 23.3 Diagonal covariance

Diagonal covariance improves stability but ignores feature correlation inside mixture components.

Full covariance could improve fit but is more fragile.

### 23.4 Fixed state and component counts

We use 4 states and 2 mixture components per state.

Real calibration should compare multiple choices using likelihood, stability, interpretability, and downstream performance.

### 23.5 EM local optima

GMM-HMM likelihoods are non-convex.

Different initialisations can lead to different regimes.

### 23.6 Feature stationarity

Tick features can change across sessions, products, contract rolls, and macro events.

Train/test stability must be checked carefully.

### 23.7 Toy strategy is not alpha

The strategy section demonstrates risk/execution conditioning only.

A real strategy needs transaction costs, queue assumptions, slippage, latency, market impact, and robust out-of-sample testing.

## 24. Research frontier and extensions

Important extensions include:

1. **Full-covariance GMM-HMM**  
   Capture correlations among microstructure features.

2. **Student-t HMM**  
   Handle heavy-tailed tick emissions more directly.

3. **Hidden semi-Markov models**  
   Explicitly model regime duration.

4. **Online EM**  
   Update regime parameters in live systems.

5. **Session-aware HMMs**  
   Separate day session, night session, open, close, and lunch break effects.

6. **Product-specific Chinese futures regimes**  
   Model different products with different tick sizes, liquidity, and night sessions.

7. **Regime-aware execution models**  
   Adjust participation rate, spread crossing, and order aggressiveness by regime.

8. **L1-to-bar regime features**  
   Aggregate tick regime probabilities into bar-level features for alpha models.

9. **LOB extension**  
   If L2/L3 data becomes available, add depth imbalance, queue imbalance, cancellation flow, and microprice.

10. **Deep sequence models**  
   Compare GMM-HMM with temporal CNNs, transformers, and neural state-space models.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_05_factor_momentum_and_volatility_scaling**  
   Use tick regimes as risk filters for trend signals.

2. **03_06_cross_sectional_alpha_features**  
   Convert regime probabilities into model features.

3. **03_07_tree_models_for_return_prediction**  
   Use regime probabilities in supervised ML.

4. **05_03_transaction_costs_slippage_latency**  
   Make execution costs regime-dependent.

5. **06_04_hawkes_process_order_flow**  
   Model quote update events and clustering.

6. **07_01_multi_asset_cta_research_pipeline**  
   Use regime overlays in a futures strategy.

7. **Chinese futures L1 microstructure capstone**  
   Build a full top-of-book regime, execution, and alpha research pipeline.

## 26. Summary

This notebook implemented GMM-HMM tick regime detection using L1 top-of-book data.

It showed how to:

1. simulate L1 bid/ask quotes with hidden microstructure regimes;
2. engineer top-of-book features without full LOB data;
3. standardise features using training data only;
4. implement diagonal Gaussian mixture emissions;
5. run log-space forward-backward inference;
6. estimate a GMM-HMM with EM;
7. sort regimes by microstructure risk;
8. compute Viterbi state paths;
9. interpret transition matrices and durations;
10. analyse mixture components;
11. filter test regimes online;
12. build a toy regime-based execution/risk filter;
13. save all outputs and diagnostics.

The key computational takeaway:

> GMM-HMMs are useful when single-Gaussian regimes are too simple for noisy tick-level features.

The key financial takeaway:

> Even with only L1 bid/ask data, regime probabilities can become useful features for execution control, risk filtering, and downstream alpha research.

## 27. Further reading

- Rabiner, "A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition."
- Hamilton, *Time Series Analysis.*
- Zucchini, MacDonald, and Langrock, *Hidden Markov Models for Time Series.*
- Literature on mixture HMMs, hidden semi-Markov models, online EM, and market microstructure regime detection.
- Literature on top-of-book quote dynamics, spread regimes, and high-frequency volatility.